# Assignment 1: RNN and LSTM from Scratch

## Overview

In this assignment, you will implement the fundamental recurrent architectures from raw tensor
operations. No `nn.RNN`, no `nn.LSTM` -- just you, matrix multiplications, and activation functions.
By building these cells by hand, you will internalize the mechanics of gating, cell state updates,
and sequential processing.

You will then train your implementations on a character-level language model and observe, empirically,
the vanishing gradient problem and how LSTM solves it.

### Learning Objectives
- Implement a vanilla RNN cell from raw tensor operations
- Implement an LSTM cell with all four gates from scratch
- Train a character-level language model with both architectures
- Empirically demonstrate vanishing gradients and LSTM's advantage

### Key Equations

**Vanilla RNN:**

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$

**LSTM Gates:**

$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f) \quad \text{(forget gate)}$$
$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i) \quad \text{(input gate)}$$
$$\tilde{C}_t = \tanh(W_g [h_{t-1}, x_t] + b_g) \quad \text{(candidate)}$$
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \quad \text{(cell state update)}$$
$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o) \quad \text{(output gate)}$$
$$h_t = o_t \odot \tanh(C_t) \quad \text{(hidden state)}$$

**Estimated time:** 8-12 hours

---
## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time
import sys
sys.path.insert(0, '../../')

from shared_utils.common import set_seed, get_device

set_seed(42)
device = get_device()
print(f"Using device: {device}")

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

### Helper Functions

In [ ]:
def generate_sequence_data(text: str, seq_length: int, batch_size: int):
    """Generate batches of (input, target) pairs for character-level language modeling.
    
    Args:
        text: The full text corpus as a string
        seq_length: Length of each sequence
        batch_size: Number of sequences per batch
        
    Yields:
        (input_batch, target_batch): Both of shape (batch_size, seq_length) as LongTensors
    """
    # Build character vocabulary
    chars = sorted(set(text))
    char_to_idx = {ch: i for i, ch in enumerate(chars)}
    idx_to_char = {i: ch for i, ch in enumerate(chars)}
    
    # Encode the text
    data = torch.tensor([char_to_idx[ch] for ch in text], dtype=torch.long)
    
    # Calculate number of complete batches
    n_batches = (len(data) - 1) // (seq_length * batch_size)
    
    # Trim data to fit complete batches
    data = data[:n_batches * seq_length * batch_size + 1]
    
    # Reshape into (batch_size, -1) for contiguous sequences per batch element
    x_data = data[:-1].view(batch_size, -1)
    y_data = data[1:].view(batch_size, -1)
    
    for i in range(0, x_data.size(1) - seq_length + 1, seq_length):
        x_batch = x_data[:, i:i+seq_length]
        y_batch = y_data[:, i:i+seq_length]
        yield x_batch, y_batch


def plot_gate_activations(gate_history: dict, title: str = "LSTM Gate Activations"):
    """Plot LSTM gate activations over time steps.
    
    Args:
        gate_history: Dict with keys 'forget', 'input', 'output', 'candidate'
                      each mapping to a list of tensors (one per time step)
        title: Plot title
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    gate_names = ['forget', 'input', 'output', 'candidate']
    
    for ax, name in zip(axes.flat, gate_names):
        if name in gate_history:
            values = [g.mean().item() for g in gate_history[name]]
            ax.plot(values)
            ax.set_title(f'{name.capitalize()} Gate')
            ax.set_xlabel('Time Step')
            ax.set_ylabel('Mean Activation')
            ax.set_ylim(-0.1, 1.1)
            ax.grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_hidden_states(hidden_history: list, title: str = "Hidden State Norms Over Time"):
    """Plot hidden state norms across time steps.
    
    Args:
        hidden_history: List of hidden state tensors, one per time step
        title: Plot title
    """
    norms = [h.norm(dim=-1).mean().item() for h in hidden_history]
    plt.figure(figsize=(10, 4))
    plt.plot(norms)
    plt.xlabel('Time Step')
    plt.ylabel('Mean Hidden State Norm')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.show()


print("Helper functions loaded.")

---
## Part 1: Implement a Vanilla RNN Cell (25%)

Implement a class `ManualRNNCell` that computes:

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$

**Requirements:**
1. Accept `input_size` and `hidden_size` as constructor arguments
2. Use Xavier initialization for weight matrices
3. Initialize biases to zero
4. Implement `forward(self, x_t, h_prev)` method
5. Implement `parameters(self)` returning all learnable tensors

In [ ]:
class ManualRNNCell:
    """Vanilla RNN cell implemented from scratch.
    
    Computes: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
    """
    
    def __init__(self, input_size: int, hidden_size: int):
        """Initialize RNN cell.
        
        Args:
            input_size: Dimensionality of input features
            hidden_size: Dimensionality of hidden state
        """
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # YOUR CODE HERE
        # Initialize W_xh (hidden_size, input_size) with Xavier initialization
        # Initialize W_hh (hidden_size, hidden_size) with Xavier initialization
        # Initialize b_h (hidden_size,) with zeros
        # All tensors must have requires_grad=True
        raise NotImplementedError("Initialize weights for ManualRNNCell")
    
    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:
        """Forward pass of the RNN cell.
        
        Args:
            x_t: Input at current time step, shape (batch_size, input_size)
            h_prev: Previous hidden state, shape (batch_size, hidden_size)
            
        Returns:
            h_t: New hidden state, shape (batch_size, hidden_size)
        """
        # YOUR CODE HERE
        raise NotImplementedError("Implement ManualRNNCell.forward")
    
    def parameters(self) -> list:
        """Return list of all learnable parameters."""
        # YOUR CODE HERE
        raise NotImplementedError("Implement ManualRNNCell.parameters")

### Test: Verify ManualRNNCell against nn.RNNCell

In [ ]:
def test_manual_rnn_cell():
    """Verify ManualRNNCell produces the same output as nn.RNNCell."""
    set_seed(42)
    input_size, hidden_size = 10, 20
    batch_size = 5
    
    # Create both cells
    manual_cell = ManualRNNCell(input_size, hidden_size)
    torch_cell = nn.RNNCell(input_size, hidden_size)
    
    # Copy weights from manual to PyTorch (or vice versa)
    with torch.no_grad():
        torch_cell.weight_ih.copy_(manual_cell.W_xh)
        torch_cell.weight_hh.copy_(manual_cell.W_hh)
        torch_cell.bias_ih.zero_()
        torch_cell.bias_hh.copy_(manual_cell.b_h)
    
    # Test inputs
    x = torch.randn(batch_size, input_size)
    h = torch.randn(batch_size, hidden_size)
    
    # Forward pass
    manual_output = manual_cell.forward(x, h)
    torch_output = torch_cell(x, h)
    
    max_diff = (manual_output - torch_output).abs().max().item()
    print(f"Max absolute difference: {max_diff:.2e}")
    assert torch.allclose(manual_output, torch_output, atol=1e-6), \
        f"RNN cell outputs do not match! Max diff: {max_diff:.2e}"
    print("PASSED: ManualRNNCell matches nn.RNNCell")

test_manual_rnn_cell()

---
## Part 2: Implement an LSTM Cell (30%)

Implement a class `ManualLSTMCell` that computes all four gates.

**Critical:** Initialize the forget gate bias to 1.0 so the LSTM can remember from the start.
If left at 0, the sigmoid outputs 0.5, meaning the cell forgets half its state at every step.

**Note on PyTorch weight layout:** PyTorch's `nn.LSTMCell` stores the gates in order
`(input_gate, forget_gate, cell_gate, output_gate)`, which differs from the textbook order.

In [ ]:
class ManualLSTMCell:
    """LSTM cell implemented from scratch with all four gates.
    
    Gates:
        f_t = sigmoid(W_f @ [h_{t-1}, x_t] + b_f)   -- forget gate
        i_t = sigmoid(W_i @ [h_{t-1}, x_t] + b_i)   -- input gate
        g_t = tanh(W_g @ [h_{t-1}, x_t] + b_g)      -- candidate
        C_t = f_t * C_{t-1} + i_t * g_t              -- cell state update
        o_t = sigmoid(W_o @ [h_{t-1}, x_t] + b_o)    -- output gate
        h_t = o_t * tanh(C_t)                        -- hidden state
    """
    
    def __init__(self, input_size: int, hidden_size: int):
        """Initialize LSTM cell.
        
        Args:
            input_size: Dimensionality of input features
            hidden_size: Dimensionality of hidden/cell state
        """
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        # YOUR CODE HERE
        # For each gate (forget, input, candidate, output):
        #   - W_*: shape (hidden_size, hidden_size + input_size), Xavier init
        #   - b_*: shape (hidden_size,)
        # IMPORTANT: Initialize forget gate bias to 1.0, all others to 0.0
        # All tensors must have requires_grad=True
        raise NotImplementedError("Initialize weights for ManualLSTMCell")
    
    def forward(
        self,
        x_t: torch.Tensor,
        h_prev: torch.Tensor,
        c_prev: torch.Tensor
    ) -> tuple:
        """Forward pass of the LSTM cell.
        
        Args:
            x_t: Input, shape (batch_size, input_size)
            h_prev: Previous hidden state, shape (batch_size, hidden_size)
            c_prev: Previous cell state, shape (batch_size, hidden_size)
            
        Returns:
            (h_t, c_t): New hidden state and cell state, both (batch_size, hidden_size)
        """
        # YOUR CODE HERE
        # 1. Concatenate h_prev and x_t along the feature dimension
        # 2. Compute each gate
        # 3. Update cell state and hidden state
        raise NotImplementedError("Implement ManualLSTMCell.forward")
    
    def parameters(self) -> list:
        """Return list of all learnable parameters."""
        # YOUR CODE HERE
        raise NotImplementedError("Implement ManualLSTMCell.parameters")

### Test: Verify ManualLSTMCell against nn.LSTMCell

In [ ]:
def test_manual_lstm_cell():
    """Verify ManualLSTMCell produces the same output as nn.LSTMCell.
    
    Note: PyTorch stores LSTM weights as a single matrix with gates in the order:
    (input_gate, forget_gate, cell_gate, output_gate)
    
    weight_ih: (4*hidden_size, input_size) -- [W_i_x; W_f_x; W_g_x; W_o_x]
    weight_hh: (4*hidden_size, hidden_size) -- [W_i_h; W_f_h; W_g_h; W_o_h]
    bias_ih + bias_hh are added together for the final bias.
    """
    set_seed(42)
    input_size, hidden_size = 10, 20
    batch_size = 5
    
    manual_cell = ManualLSTMCell(input_size, hidden_size)
    torch_cell = nn.LSTMCell(input_size, hidden_size)
    
    # Copy weights from manual cell to PyTorch's cell
    # You need to map your separate gate weights to PyTorch's concatenated format
    # PyTorch order: input_gate (i), forget_gate (f), cell_gate (g), output_gate (o)
    with torch.no_grad():
        # YOUR CODE HERE: Copy weights from manual_cell to torch_cell
        # Hint: torch_cell.weight_ih has shape (4*hidden_size, input_size)
        #       torch_cell.weight_hh has shape (4*hidden_size, hidden_size)
        #       The gate order is [i, f, g, o]
        pass
    
    # Test inputs
    x = torch.randn(batch_size, input_size)
    h = torch.randn(batch_size, hidden_size)
    c = torch.randn(batch_size, hidden_size)
    
    manual_h, manual_c = manual_cell.forward(x, h, c)
    torch_h, torch_c = torch_cell(x, (h, c))
    
    h_diff = (manual_h - torch_h).abs().max().item()
    c_diff = (manual_c - torch_c).abs().max().item()
    print(f"Max h difference: {h_diff:.2e}")
    print(f"Max c difference: {c_diff:.2e}")
    
    assert torch.allclose(manual_h, torch_h, atol=1e-6), f"h mismatch! Max diff: {h_diff:.2e}"
    assert torch.allclose(manual_c, torch_c, atol=1e-6), f"c mismatch! Max diff: {c_diff:.2e}"
    print("PASSED: ManualLSTMCell matches nn.LSTMCell")

test_manual_lstm_cell()

---
## Part 3: Character-Level Language Model (25%)

Build a character-level language model: given a sequence of characters, predict the next character.
Train with both ManualRNNCell and ManualLSTMCell.

### Dataset
Download Shakespeare text (~1.1M characters) or use any plain text of at least 100K characters.

In [ ]:
# Download or load a text dataset
# Option 1: Download Shakespeare
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    text = urllib.request.urlopen(url).read().decode('utf-8')
    print(f"Downloaded Shakespeare text: {len(text):,} characters")
except:
    # Fallback: generate synthetic text if download fails
    print("Download failed. Using synthetic data.")
    text = "hello world. " * 10000

# Build vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print(f"First 100 characters: {text[:100]}")

### Model Architecture

Build a wrapper model:
- Embedding layer: `nn.Embedding(vocab_size, embed_size)`
- Your manual RNN or LSTM cell, applied across all time steps in a loop
- Output projection: linear layer mapping hidden_size to vocab_size
- Cross-entropy loss

In [ ]:
class CharLanguageModel(nn.Module):
    """Character-level language model using a manual RNN or LSTM cell."""
    
    def __init__(self, vocab_size: int, embed_size: int, hidden_size: int, cell_type: str = 'rnn'):
        """Initialize the language model.
        
        Args:
            vocab_size: Number of unique characters
            embed_size: Embedding dimension
            hidden_size: Hidden state dimension
            cell_type: 'rnn' or 'lstm'
        """
        super().__init__()
        self.hidden_size = hidden_size
        self.cell_type = cell_type
        
        # YOUR CODE HERE
        # 1. Create nn.Embedding(vocab_size, embed_size)
        # 2. Create ManualRNNCell or ManualLSTMCell depending on cell_type
        # 3. Create nn.Linear(hidden_size, vocab_size) for output projection
        raise NotImplementedError("Implement CharLanguageModel.__init__")
    
    def forward(self, x: torch.Tensor, hidden=None) -> tuple:
        """Forward pass through the language model.
        
        Args:
            x: Input token indices, shape (batch_size, seq_length)
            hidden: Initial hidden state (and cell state for LSTM)
            
        Returns:
            logits: Output logits, shape (batch_size, seq_length, vocab_size)
            hidden: Final hidden state (and cell state for LSTM)
        """
        # YOUR CODE HERE
        # 1. Embed the input tokens
        # 2. Loop through time steps, applying the cell at each step
        # 3. Collect hidden states and project to vocab_size
        raise NotImplementedError("Implement CharLanguageModel.forward")

### Training Loop

In [ ]:
def train_model(
    model: nn.Module,
    text: str,
    char_to_idx: dict,
    n_epochs: int = 20,
    seq_length: int = 50,
    batch_size: int = 64,
    lr: float = 1e-3,
    max_norm: float = 5.0,
    log_interval: int = 100
) -> list:
    """Train the character-level language model.
    
    Implements truncated BPTT with gradient clipping.
    
    Args:
        model: CharLanguageModel instance
        text: Training text
        char_to_idx: Character-to-index mapping
        n_epochs: Number of training epochs
        seq_length: Sequence length for BPTT
        batch_size: Batch size
        lr: Learning rate
        max_norm: Max gradient norm for clipping
        log_interval: Print loss every N batches
        
    Returns:
        losses: List of (epoch, batch, loss) tuples
    """
    # YOUR CODE HERE
    # 1. Use Adam optimizer
    # 2. For each epoch, iterate over batches from generate_sequence_data
    # 3. Apply truncated BPTT: detach hidden state between chunks
    # 4. Apply gradient clipping with max_norm
    # 5. Log training loss every log_interval batches
    raise NotImplementedError("Implement train_model")

In [ ]:
# Train RNN model
print("Training RNN model...")
rnn_model = CharLanguageModel(vocab_size, embed_size=64, hidden_size=128, cell_type='rnn')
rnn_losses = train_model(rnn_model, text, char_to_idx, n_epochs=20, seq_length=50)

In [ ]:
# Train LSTM model
print("Training LSTM model...")
lstm_model = CharLanguageModel(vocab_size, embed_size=64, hidden_size=128, cell_type='lstm')
lstm_losses = train_model(lstm_model, text, char_to_idx, n_epochs=20, seq_length=50)

### Comparison: RNN vs LSTM at Different Sequence Lengths

In [ ]:
# Train both models at sequence lengths 25, 50, 100, 200
# Record final losses for comparison
seq_lengths = [25, 50, 100, 200]
results = {'rnn': {}, 'lstm': {}}

for seq_len in seq_lengths:
    print(f"\n{'='*50}")
    print(f"Sequence length: {seq_len}")
    print(f"{'='*50}")
    
    for cell_type in ['rnn', 'lstm']:
        print(f"\nTraining {cell_type.upper()} with seq_length={seq_len}...")
        model = CharLanguageModel(vocab_size, embed_size=64, hidden_size=128, cell_type=cell_type)
        losses = train_model(model, text, char_to_idx, n_epochs=10, seq_length=seq_len)
        results[cell_type][seq_len] = losses

In [ ]:
# Plot training curves for all configurations
fig, axes = plt.subplots(1, len(seq_lengths), figsize=(20, 5), sharey=True)

for ax, seq_len in zip(axes, seq_lengths):
    for cell_type, color in [('rnn', 'blue'), ('lstm', 'red')]:
        if seq_len in results[cell_type]:
            losses = results[cell_type][seq_len]
            loss_values = [l[2] for l in losses]  # Extract loss from (epoch, batch, loss)
            ax.plot(loss_values, label=cell_type.upper(), color=color, alpha=0.7)
    ax.set_title(f'Seq Length = {seq_len}')
    ax.set_xlabel('Training Step')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('Loss')
plt.suptitle('RNN vs LSTM: Training Loss at Different Sequence Lengths', fontsize=14)
plt.tight_layout()
plt.show()

### Generate Text Samples

In [ ]:
def generate_text(
    model: nn.Module,
    start_str: str,
    char_to_idx: dict,
    idx_to_char: dict,
    length: int = 500,
    temperature: float = 0.8
) -> str:
    """Generate text from a trained character-level model.
    
    Args:
        model: Trained CharLanguageModel
        start_str: Seed string to start generation
        char_to_idx: Character-to-index mapping
        idx_to_char: Index-to-character mapping
        length: Number of characters to generate
        temperature: Sampling temperature (higher = more random)
        
    Returns:
        Generated text string
    """
    model.eval()
    hidden = None
    generated = list(start_str)
    
    # Feed in the start string
    input_seq = torch.tensor([[char_to_idx[ch] for ch in start_str]], dtype=torch.long)
    with torch.no_grad():
        logits, hidden = model(input_seq, hidden)
    
    # Generate one character at a time
    current_char_idx = torch.tensor([[char_to_idx[start_str[-1]]]], dtype=torch.long)
    for _ in range(length):
        with torch.no_grad():
            logits, hidden = model(current_char_idx, hidden)
        
        # Apply temperature and sample
        probs = F.softmax(logits[0, -1] / temperature, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1).item()
        generated.append(idx_to_char[next_idx])
        current_char_idx = torch.tensor([[next_idx]], dtype=torch.long)
    
    return ''.join(generated)

# Generate samples from both models
print("=" * 60)
print("RNN Generated Text:")
print("=" * 60)
print(generate_text(rnn_model, "The ", char_to_idx, idx_to_char))

print("\n" + "=" * 60)
print("LSTM Generated Text:")
print("=" * 60)
print(generate_text(lstm_model, "The ", char_to_idx, idx_to_char))

---
## Part 4: Gradient Analysis (20%)

Empirically demonstrate the vanishing gradient problem in RNNs and show how LSTM mitigates it.

### Gradient Norm Tracking

During training, record the gradient norm of the loss with respect to the hidden state at each
time step. After `loss.backward()`, examine the `.grad` attribute of each hidden state.

In [ ]:
def analyze_gradients(model: nn.Module, text: str, char_to_idx: dict, seq_length: int = 100):
    """Analyze gradient flow through time in an RNN/LSTM model.
    
    Args:
        model: CharLanguageModel instance
        text: Training text
        char_to_idx: Character-to-index mapping  
        seq_length: Length of the sequence to analyze
        
    Returns:
        grad_norms: List of gradient norms at each time step (from the loss backward)
    """
    # YOUR CODE HERE
    # 1. Create a single input sequence of length seq_length
    # 2. Run forward pass, storing hidden states with retain_grad()
    # 3. Compute loss
    # 4. Call loss.backward()
    # 5. Extract gradient norms from each hidden state
    raise NotImplementedError("Implement analyze_gradients")

In [ ]:
# Analyze gradients for both models
rnn_grads = analyze_gradients(rnn_model, text, char_to_idx, seq_length=100)
lstm_grads = analyze_gradients(lstm_model, text, char_to_idx, seq_length=100)

# Plot: Vanishing Gradients: RNN vs LSTM
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(rnn_grads, color='blue')
ax1.set_title('Vanilla RNN: Gradient Norms vs Time Step')
ax1.set_xlabel('Time Step (distance from loss)')
ax1.set_ylabel('Gradient Norm')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

ax2.plot(lstm_grads, color='red')
ax2.set_title('LSTM: Gradient Norms vs Time Step')
ax2.set_xlabel('Time Step (distance from loss)')
ax2.set_ylabel('Gradient Norm')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.suptitle('Vanishing Gradients: RNN vs LSTM', fontsize=14)
plt.tight_layout()
plt.show()

### Gradient Clipping Experiment

In [ ]:
# Train RNN WITHOUT gradient clipping to observe exploding gradients
# Then train WITH gradient clipping and compare

# YOUR CODE HERE
# 1. Train RNN with max_norm=None (no clipping)
# 2. Train RNN with max_norm=5.0 (clipping)
# 3. Record gradient norms before clipping throughout training
# 4. Plot gradient norms over training steps

### Written Analysis

Write 1-2 paragraphs interpreting your results:
- Why does the LSTM outperform the RNN on longer sequences?
- What do the gradient plots tell you?

**Your analysis here:**

YOUR ANSWER HERE

---
## Stretch Goals (Optional)

1. **GRU cell from scratch** -- Add it to the comparison experiments.
2. **Find the RNN failure point** -- What sequence length causes complete failure?
3. **Orthogonal initialization** for `W_hh` -- Compare to Xavier.
4. **Temperature sampling** -- Show temperature's effect on diversity vs coherence.
5. **Layer normalization** within the LSTM cell.

In [ ]:
# Stretch goal workspace

def sample_with_temperature(logits, temperature=1.0):
    """Sample from logits with temperature scaling."""
    scaled_logits = logits / temperature
    probs = torch.softmax(scaled_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

# YOUR STRETCH GOAL CODE HERE